# Inverse modeling with STILT, in Python

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmineau/stilt-inversion-demo/blob/main/inverse_modeling_with_stilt.ipynb)

**James Mineau**, Lin group, University of Utah

Three small, composable Python packages that cover the STILT inverse-modeling
pipeline from meteorology to posterior flux, plus a look at the Salt Lake Valley
methane inversion they were built for. This whole notebook runs on a fresh Colab;
everything installs and downloads itself.

| stage | package | what it does |
|---|---|---|
| meteorology | [`arlmet`](https://github.com/jmineau/arl-met) | reads/writes the ARL-packed met that HYSPLIT and STILT use, as `xarray`, and fetches it from NOAA |
| transport | [PYSTILT](https://github.com/jmineau/PYSTILT) | runs STILT in Python (pip `pystilt`, import `stilt`), producing trajectories and footprints |
| inversion | [`fips`](https://github.com/jmineau/fips) | assembles and solves the linear Bayesian inverse problem |
| application | [`slv`](https://github.com/jmineau/slv) | an example of wrapping the pystilt and fips pipeline (SLV CH4) |

The footprint is the hinge. It is STILT's output, and it is also a row of the
inverse problem's Jacobian. Staying in one (Python) ecosystem means you never
leave `xarray` and `pandas` from the met file to the posterior.

## Setup

Installs the stack from PyPI and pulls the small demo data. `fips[flux]` brings in
`pystilt` (and `arlmet`, `cartopy`, `matplotlib`); `arlmet[sources]` adds the
`s3fs` needed to fetch met from NOAA. This takes a minute or two on a fresh Colab.

In [ ]:
%pip install -q "fips[flux]" "arlmet[sources]"

In [ ]:
import os, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# Pull the small demo inputs (a footprint and a trajectory sample). They are the
# fallback used if a given Colab cannot run the live STILT run further down.
Path("data").mkdir(exist_ok=True)
for f in ["demo_footprint.nc", "demo_trajectories.parquet"]:
    if not Path("data", f).exists():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/jmineau/stilt-inversion-demo/main/data/" + f, f"data/{f}")

print("ready")

## The problem

A tower measures a CH4 enhancement $z$ above background. We want the upwind
surface fluxes $x$ that produced it. STILT gives the footprint $H$, the
sensitivity of the measurement to each surface grid cell, and we invert

$$ z = Hx + c + \varepsilon $$

for $x$ given a prior $x_0$ and error budgets. Everything below is the plumbing
from a met file to $\hat{x}$.

---
# 1. `arlmet`: meteorology for STILT

STILT and HYSPLIT run on ARL-packed binary met, a NOAA format. Historically you
wrangle it with Fortran utilities. `arlmet` reads and writes it as a lazy
`xarray.Dataset`, and it can fetch the met straight from NOAA's public archive,
cropped to your domain on download.

In [ ]:
import arlmet

# Fetch a domain-cropped HRRR from NOAA's anonymous S3 archive (a real download).
# Falls back to the stilt-tutorials met if S3 or s3fs is not reachable in your Colab.
Path("met").mkdir(exist_ok=True)
try:
    from arlmet.sources import HRRRSource
    files = HRRRSource().fetch("2024-07-18T00", "2024-07-18T01", local_dir="met",
                               bbox=(-114.0, 39.0, -110.5, 42.5))   # Utah
    met = files[0]
    print("fetched live from NOAA S3:", Path(met).name)
except Exception as ex:
    print(f"S3 fetch unavailable ({type(ex).__name__}), using stilt-tutorials met")
    met = "met/20151210.18z.hrrra"
    urllib.request.urlretrieve("https://github.com/uataq/stilt-tutorials/raw/main/01-wbb/met/20151210.18z.hrrra", met)

ds = arlmet.open_dataset(met)        # lazy: nothing is unpacked yet
ds

A multi-gigabyte file opens instantly because nothing is unpacked until you ask
for it. And because `arlmet` parses the ARL index records, the dataset arrives
self-describing: the fields split into surface and upper-air, and it carries its
native grid and map projection (as a `pyproj` CRS) on the `.arl` accessor.

In [ ]:
# Variables split by whether they carry a vertical "level" dimension.
sfc = [v for v in ds.data_vars if "level" not in ds[v].dims]   # surface fields
ua  = [v for v in ds.data_vars if "level" in ds[v].dims]       # upper-air fields
print(f"surface fields ({len(sfc)}): " + ", ".join(sfc))
print(f"upper-air fields ({len(ua)}): " + ", ".join(ua))

# The .arl accessor exposes the ARL metadata arlmet parsed from the index records.
g = ds.arl.grid
print(f"\nsource   : {ds.arl.source}")
print(f"grid     : {g.nx} x {g.ny}, lat/lon native = {g.is_latlon}")
print(f"crs      : {g.crs.to_string()[:60]}")   # a real pyproj CRS
print(f"vertical : {ds.arl.vertical_axis}")

Let us pull a field STILT users care about, the PBL height, and map it.

In [ ]:
import cartopy.crs as ccrs

pblh = ds["PBLH"].isel(time=0)
ax = plt.axes(projection=ccrs.PlateCarree())
pblh.plot(x="lon", y="lat", cmap="viridis", ax=ax,
          cbar_kwargs={"label": "PBL height [m]"})
ax.coastlines(); ax.gridlines(draw_labels=True)
ax.plot(-111.85, 40.77, "r^", ms=11, transform=ccrs.PlateCarree(), label="UOU")
ax.legend(loc="upper right"); ax.set_title(f"HRRR PBL height, {pblh.time.values}")
plt.show()

That was a surface field. ARL files also pack the full column, and `sample_points`
interpolates any field to whatever heights you ask for. We sample a stack of
heights at the tower to pull a temperature and wind-speed profile, no manual
level-wrangling required.

In [ ]:
# A column at the tower: one lon/lat, a stack of heights, one time.
uou_lon, uou_lat = -111.848, 40.766
heights = np.arange(50, 3001, 200)               # m above ground
col = pd.DataFrame({"lon": uou_lon, "lat": uou_lat, "z": heights,
                    "time": ds["time"].values[0]})

# sample_points interpolates each field to those heights (z_kind="agl").
prof = arlmet.sample_points(arlmet.File(met), col, ["TEMP", "UWND", "VWND"], z_kind="agl")
prof["wspd"] = np.hypot(prof["UWND"], prof["VWND"])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 5), sharey=True)
a1.plot(prof["TEMP"] - 273.15, prof["z"], "o-", color="tab:red")
a1.set(xlabel="temperature (C)", ylabel="height above ground (m)", title="temperature")
a2.plot(prof["wspd"], prof["z"], "s-", color="tab:blue")
a2.set(xlabel="wind speed (m/s)", title="wind speed")
fig.suptitle("arlmet vertical profile at the UOU tower"); plt.show()

You can also sample any field at points, exactly what you need along a trajectory
or at a receptor.

In [ ]:
# One row per query point (lon, lat, z, time); sample_points returns the points
# with the requested field(s) interpolated to each. This is the primitive for
# sampling met along a trajectory or at a receptor.
pts = pd.DataFrame({"lon": [-111.85], "lat": [40.77], "z": [10.0],
                    "time": [ds["time"].values[0]]})
arlmet.sample_points(arlmet.File(met), pts, ["PBLH"])

And `arlmet` is more than a reader. The same package fetches met from a catalog
of NOAA archives (cropping before it unpacks), concatenates and subsets ARL
files, and writes ARL back out, so the whole met side of a STILT run can stay in
Python rather than a stack of Fortran utilities.

In [ ]:
# arlmet ships one source class per NOAA archive; each can fetch and crop on download.
from arlmet import sources
archives = [n[:-6] for n in dir(sources)
            if n.endswith("Source") and n != "MeteorologySource"]
print("NOAA archives arlmet can fetch:\n  " + ", ".join(archives))
print("\nARL read / write / reshape:\n  open_dataset, write_dataset, "
      "extract_subset, concat, concat_by_time, sample_points")

For example, `extract_subset` crops an ARL file to a smaller domain by decoding
only the tiles it needs. We crop the met to its inner quarter, then plot a surface
field with the cropped region outlined. On a projected (non-latlon) grid the crop
is a block of native cells, so its outline is not a lon/lat square.

In [ ]:
import cartopy.crs as ccrs

# A bbox (lon/lat box) over the inner quarter of the file's domain.
lon_lo, lon_hi = float(ds.lon.min()), float(ds.lon.max())
lat_lo, lat_hi = float(ds.lat.min()), float(ds.lat.max())
mx, my = (lon_hi - lon_lo) * 0.25, (lat_hi - lat_lo) * 0.25
bbox = (lon_lo + mx, lat_lo + my, lon_hi - mx, lat_hi - my)

arlmet.extract_subset(met, "met_crop.arl", bbox=bbox)        # decodes only the bbox tiles
cropped = arlmet.open_dataset("met_crop.arl")
print(f"full grid: {ds.arl.grid.nx} x {ds.arl.grid.ny}  ->  "
      f"cropped: {cropped.arl.grid.nx} x {cropped.arl.grid.ny}")

# A surface field to look at, and the cropped grid's outline traced in lon/lat.
field = next((v for v in ["PRSS", "T02M", "PBLH"] if v in ds), list(ds.data_vars)[0])
clon, clat = cropped.lon.values, cropped.lat.values
if clon.ndim == 2:                 # projected grid: trace the cell-block perimeter
    ox = np.concatenate([clon[0], clon[:, -1], clon[-1, ::-1], clon[::-1, 0]])
    oy = np.concatenate([clat[0], clat[:, -1], clat[-1, ::-1], clat[::-1, 0]])
else:                              # lat/lon grid: an axis-aligned box
    ox = [clon.min(), clon.max(), clon.max(), clon.min(), clon.min()]
    oy = [clat.min(), clat.min(), clat.max(), clat.max(), clat.min()]

ax = plt.axes(projection=ccrs.PlateCarree())
ds[field].isel(time=0).plot(x="lon", y="lat", cmap="viridis", ax=ax,
                            cbar_kwargs={"label": field})
ax.plot(ox, oy, "r-", lw=2.5, transform=ccrs.PlateCarree(), label="extract_subset region")
ax.coastlines(); ax.legend(loc="upper right")
ax.set_title(f"{field} with the extract_subset region")
plt.show()

---
# 2. PYSTILT: run STILT in Python

PYSTILT is a from-scratch Python implementation of STILT. The names take a second:
the project is PYSTILT, you `pip install pystilt`, and you `import stilt`. It
bundles the HYSPLIT engine, so that one install is all you need to actually run
the model. There are two ways to drive it: interactively for one-off trajectories
and footprints, or from the command line with a `config.yaml` for reproducible
batches.

First, grab a day of UOU-domain HRRR (the December 2015 PCAP case from the
[`stilt-tutorials`](https://github.com/uataq/stilt-tutorials) repo) for the run.

In [ ]:
# Grab the four 6-hourly HRRR files for the case day (cached after first download).
Path("met_run").mkdir(exist_ok=True)
for h in ["00", "06", "12", "18"]:
    f = f"20151210.{h}z.hrrra"
    if not Path("met_run", f).exists():
        urllib.request.urlretrieve("https://github.com/uataq/stilt-tutorials/raw/main/01-wbb/met/" + f, f"met_run/{f}")

print("met ready:", sorted(os.listdir("met_run")))

### Interactive: a one-off footprint

Define a receptor, point it at the met, choose a footprint grid, and run. This
takes about half a minute (it is really running HYSPLIT under the hood).

In [ ]:
import stilt

# The config: how far back to run, how many particles, which met, and the
# footprint grid to accumulate onto.
config = stilt.ModelConfig(
    n_hours=-12, numpar=100,
    mets={"hrrr": stilt.MetConfig(directory="met_run",
                                  file_format="%Y%m%d.%Hz.hrrra", file_tres="6h")},
    footprints={"default": stilt.FootprintConfig(
        grid=stilt.Grid(xmin=-112.5, xmax=-111.5, ymin=40.4, ymax=41.0,
                        xres=0.01, yres=0.01))},
)

# The receptor: where and when the measurement was taken (the UOU tower).
receptor = stilt.PointReceptor(time=pd.Timestamp("2015-12-10 18:00"),
                               longitude=-111.848, latitude=40.766, altitude=10.0)

# Run it. If HYSPLIT cannot run in this Colab, fall back to the bundled result.
try:
    model = stilt.Model(project="demo_run", receptors=[receptor], config=config)
    model.run().wait()
    [sim] = model.simulations.values()
    foot, traj = sim.get_footprint("default"), sim.trajectories
    print("ran STILT live:", dict(foot.data.sizes))
except Exception as ex:
    print(f"live run unavailable ({type(ex).__name__}); loading the bundled result")
    foot = stilt.Footprint.from_netcdf("data/demo_footprint.nc")
    traj = stilt.Trajectories.from_parquet("data/demo_trajectories.parquet")

The footprint: where the air the tower sampled touched the surface (ppm per
umol m-2 s-1). This is the measurement's sensitivity to surface flux.

In [ ]:
from cartopy.io.img_tiles import GoogleTiles
sat = GoogleTiles(style="satellite")           # Google satellite basemap, reused below

foot.plot.map(tiler=sat, tiler_zoom=10)         # time-integrated footprint over the SLV
plt.show()

It is built from back-trajectories released at the receptor and run backward
through the winds. Where each particle ends up is where the air entered the
domain, the endpoints, which is how we build the inversion's background term $c$
(by sampling a global field like CarbonTracker there).

In [ ]:
traj.plot.map(tiler=sat, tiler_zoom=10); plt.show()
traj.endpoints().head()    # one row per particle: the far end of its path

The trajectory table is rich: each particle carries its height, the modeled mixing
height, pressure, density, and turbulence terms at every step. The plot tracks all
the particle heights backward from the receptor (gray), with the mean mixing height
(red) for scale. Particles are released at the tower and disperse upward as they age;
the ones still inside the mixing layer are the ones touching the surface and building
the footprint.

In [ ]:
df = traj.data
fig, ax = plt.subplots(figsize=(8, 4))
for _, g in df.groupby("indx"):
    ax.plot(g["time"] / 60, g["zagl"], color="0.6", lw=0.3, alpha=0.5)
mlht = df.groupby("time")["mlht"].mean()
ax.plot(mlht.index / 60, mlht.values, "r-", lw=2, label="mean mixing height")
ax.set(xlabel="hours back from receptor", ylabel="height above ground [m]",
       title="particle heights along the back-trajectories")
ax.legend(); plt.show()

You can also reshape the particles before they become a footprint. A
`ParticleTransform` reweights the particle table; `pystilt` ships a vertical-operator
(averaging-kernel) transform and a first-order lifetime decay, declarable on a
`FootprintConfig` or applied directly. Here is lifetime decay on our particles. CH4
itself lasts about 9 years, so we use a short 3 hour lifetime to make the effect
visible: it knocks down the older particles, as a reactive species would.

In [ ]:
from stilt.config import FirstOrderLifetimeTransformSpec
from stilt.transforms import build_particle_transform

# Build the transform from a declarative spec and apply it to the particle table.
spec    = FirstOrderLifetimeTransformSpec(kind="first_order_lifetime", lifetime_hours=3.0)
decayed = build_particle_transform(spec).apply(traj.data)

# Total footprint contribution per hour-back bin, before vs after the decay.
age_h = (-traj.data["time"] / 60).round()
orig = traj.data.groupby(age_h)["foot"].sum()
dec  = decayed.groupby(age_h)["foot"].sum()

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(orig.index, orig.values, "o-", color="0.5", label="original")
ax.plot(dec.index, dec.values, "s-", color="crimson", label="after 3 h lifetime decay")
ax.set(xlabel="hours back from receptor", ylabel="total footprint contribution",
       title="a particle transform: lifetime decay shrinks the older footprint",
       yscale="log")
ax.legend(); plt.show()

### More than a single point

A tower is a `PointReceptor`, but the same `Model` accepts a `ColumnReceptor` (a
vertical column, for satellite or aircraft-profile work, the X-STILT use case)
and a `MultiPointReceptor` (arbitrary points, such as an aircraft track). Same
engine, same footprints, different sampling geometry.

In [ ]:
t = pd.Timestamp("2015-12-10 18:00")

tower  = stilt.PointReceptor(t, -111.848, 40.766, 10.0)                      # one point, one altitude
column = stilt.ColumnReceptor(t, -111.848, 40.766, bottom=0.0, top=3000.0)   # a vertical column
track  = stilt.MultiPointReceptor(t, [-111.90, -111.85, -111.80],            # several points at once
                                     [40.70, 40.75, 40.80], [50.0, 500.0, 1500.0])

for r in (tower, column, track):
    print(f"{type(r).__name__:20s} id = {r.id}")

### From raw receptors to an observation layer

Raw receptors only say *where and when* to release particles. The
`stilt.observations` layer adds the X-STILT-style measurement abstraction on top:
a `Sensor` normalizes measurements into `Observation`s, builds the right receptor
type for each, groups soundings into retrieval `Scene`s, and carries a
`VerticalOperator` (an averaging kernel) that reweights footprints. Below we define
a tower and a satellite-column sensor, make observations, build their receptors,
group the satellite soundings into overpasses, and apply an averaging kernel, all
in memory (no model run).

In [ ]:
from stilt.observations import (
    ColumnSensor, PointSensor, VerticalOperator,
    apply_vertical_operator, group_scenes_by_time_gap,
)
from stilt.observations.geometry import HorizontalGeometry

# 1. Define sensors: a fixed tower (releases at one height) and a satellite column
#    sounder (releases over a vertical layer).
tower_sensor = PointSensor(name="uou_tower", supported_species=("CH4",), default_height=36.0)
sat_sensor   = ColumnSensor(name="tropomi", supported_species=("CH4",),
                            mode="vertical", bottom=0.0, top=12000.0, altitude_ref="agl")

# 2. Build normalized Observations (the sensor stamps its name and checks species).
tower_obs = [
    tower_sensor.make_observation(time="2024-01-15 18:00", latitude=40.766,
                                  longitude=-111.847, value=2.05, units="ppm"),
]
# Three satellite retrievals: two in one overpass, one ~90 min later.
sat_obs = [
    sat_sensor.make_observation(time="2024-01-15 20:05", latitude=40.70, longitude=-111.95,
        value=1.89, units="ppm",
        geometry=HorizontalGeometry(kind="swath_cell", center_latitude=40.70,
                                    center_longitude=-111.95, swath=1201)),
    sat_sensor.make_observation(time="2024-01-15 20:06", latitude=40.78, longitude=-111.88,
        value=1.92, units="ppm",
        geometry=HorizontalGeometry(kind="swath_cell", center_latitude=40.78,
                                    center_longitude=-111.88, swath=1201)),
    sat_sensor.make_observation(time="2024-01-15 21:38", latitude=40.83, longitude=-111.80,
        value=1.95, units="ppm",
        geometry=HorizontalGeometry(kind="swath_cell", center_latitude=40.83,
                                    center_longitude=-111.80, swath=1342)),
]

# 3. Each sensor turns an Observation into the right receptor type.
print("tower    ->", tower_sensor.build_receptor(tower_obs[0]).id)
print("satellite ->", sat_sensor.build_receptor(sat_obs[0]).id)

# 4. Group soundings into overpass Scenes (raw receptors cannot express this).
scenes = group_scenes_by_time_gap(sat_obs, max_gap="30min", scene_prefix="tropomi")
print(f"\n{len(sat_obs)} soundings -> {len(scenes)} overpass scene(s):")
for s in scenes:
    print(f"   {s.id}: {len(s.observations)} obs")

# 5. Apply an averaging-kernel vertical operator to a (mock) column footprint.
ak = VerticalOperator(mode="ak", levels=[0.0, 3000.0, 12000.0], values=[1.0, 0.6, 0.2])
particles = pd.DataFrame({"indx": [1, 2, 3, 4],
                          "xhgt": [100.0, 3000.0, 8000.0, 12000.0],   # release height AGL
                          "foot": [1.0, 1.0, 1.0, 1.0]})              # unweighted
weighted = apply_vertical_operator(particles, ak, coordinate="xhgt")
print("\nAK-weighted footprint by release height:")
print(weighted[["xhgt", "foot_before_weight", "foot"]].to_string(index=False))

### No project needed: run in a temp directory

For a quick one-off you do not have to set up a project at all. Leave `project`
unset and `pystilt` runs in a temporary working directory: it computes the
footprint and leaves nothing behind. Name a project only when you want to keep the
outputs.

In [ ]:
# Same receptor and config as above, but no project= argument.
ephemeral = stilt.Model(receptors=[receptor], config=config)
print("auto project name :", ephemeral.project)
print("working dir (temp):", ephemeral.layout.output_root)
# ephemeral.run().wait() would compute here, then you pull the footprint out.

### Command line: config-driven

The same run can be driven from a `config.yaml`. `stilt init` scaffolds a project
(a `config.yaml` and a `receptors.csv`); you edit the met directory and receptors,
then `stilt run`.

In [ ]:
!stilt init demo_cli
print("\n\n--- demo_cli/config.yaml ---\n")
!sed -n '1,30p' demo_cli/config.yaml
# Then: point mets.hrrr.directory at your met, add rows to receptors.csv, and run:
#   !stilt run demo_cli

### One run, many outputs

A config is more than a met directory and a grid. One `Model` run can produce
several footprint grids at once (a fine grid for the city, a coarse grid for the
basin), and turning on the wind-error parameters runs a perturbed ensemble that
yields a transport-error footprint alongside each footprint. Same declarative
object, and it round-trips to and from a `config.yaml`.

In [ ]:
cfg = stilt.ModelConfig(
    n_hours=-24, numpar=500,
    mets={"hrrr": stilt.MetConfig(directory="met_run",
                                  file_format="%Y%m%d.%Hz.hrrra", file_tres="6h")},
    footprints={
        "city":  stilt.FootprintConfig(grid=stilt.Grid(
                     xmin=-112.3, xmax=-111.6, ymin=40.4, ymax=41.0,
                     xres=0.005, yres=0.005)),
        "basin": stilt.FootprintConfig(grid=stilt.Grid(
                     xmin=-112.5, xmax=-111.5, ymin=40.3, ymax=41.1,
                     xres=0.05, yres=0.05), error=True),
    },
    siguverr=1.8, tluverr=60.0, horcoruverr=40.0, zcoruverr=300.0,  # wind-error ensemble
)
print("footprint grids in one run:", list(cfg.footprints))
print("transport-error footprint on 'basin':", cfg.footprints["basin"].error)
print("wind error (sigma m/s, tau min, horiz km, vert m):",
      cfg.siguverr, cfg.tluverr, cfg.horcoruverr, cfg.zcoruverr)

cfg.to_yaml("demo_config.yaml")                              # write it out ...
cfg_again = stilt.ModelConfig.from_yaml("demo_config.yaml")  # ... and read it back
print("\nsaved and reloaded via from_yaml ->", list(cfg_again.footprints), "grids")
print("first lines of demo_config.yaml:")
print("\n".join(open("demo_config.yaml").read().splitlines()[:7]))

### STILT-R vs pystilt

You just ran STILT in Python. How does that compare to the established R
implementation, [STILT-R](https://github.com/uataq/stilt)?

It is the same model. `pystilt` drives the same HYSPLIT engine, and its footprints
match STILT-R numerically (`rtol=1e-7` per cell in the test suite, against a pinned
upstream commit). For the SLV work we went further and ran a formal equivalence
test on 200 receptors: footprints agree to within about 3%, which is inside STILT's
own run-to-run stochastic floor of about 4%, with spatial correlation 0.998. It is
not a different model. It is STILT, in Python.

So this is not about results, they match. It is about where the model lives:

| | STILT-R | pystilt |
|---|---|---|
| install | the `uataq` R package, then `uataq::stilt_init()` (binaries bundled); needs R, NetCDF, GDAL | `pip install pystilt` (HYSPLIT bundled) |
| drive it | edit `run_stilt.r`, or a CLI | Python objects, a `config.yaml`, or the `stilt` CLI |
| outputs | trajectories `.rds`, footprints NetCDF (CF) | trajectories Parquet, footprints NetCDF (CF) |
| ecosystem | R (`dplyr`, `raster`, `rslurm`) | `xarray` / `pandas`, and `fips`, `arlmet`, ... |
| execution | local or SLURM (`rslurm`) | local, SLURM, queue workers, Kubernetes, streaming |
| tested | basic smoke tests | typed (pyright), unit and fidelity tests |

The point is not to replace STILT-R, but to put STILT where the rest of your
Python analysis already lives. The footprints are CF NetCDF either way; what
changes is the trajectories (Parquet, not `.rds`) and the surrounding ecosystem.

**The hinge.** Time-integrate that footprint, line it up with a flux grid, and it
becomes one row of the Jacobian $H$. Stack a footprint per observation and you
have the forward operator for the inverse problem, which is where `fips` comes in.

---
# 3. `fips`: the Bayesian inverse problem

`fips` solves the linear Bayesian inverse problem

$$ z = Hx + c + \varepsilon,\quad \varepsilon\sim\mathcal N(0,S_z),\quad x\sim\mathcal N(x_0,S_0) $$

with the closed-form posterior

$$ \hat{x} = x_0 + K(z - Hx_0 - c),\qquad K = S_0 H^\top (H S_0 H^\top + S_z)^{-1}. $$

You give it the obs $z$, prior $x_0$, forward operator $H$, prior error $S_0$, and
model-data mismatch $S_z$. It returns the posterior, its uncertainty, and the
averaging kernel $A=KH$, whose trace is the DOFS (how many independent things the
data actually constrained). Here it is on a tiny synthetic problem.

In [ ]:
from fips import InverseProblem

# A small flux grid: 6 x 6 lat/lon cells. The truth has two smooth source blobs.
lats = np.round(np.arange(40.5, 41.05, 0.1), 2)
lons = np.round(np.arange(-112.1, -111.55, 0.1), 2)
cells = pd.MultiIndex.from_product([lats, lons], names=["lat", "lon"])
nlat, nlon, n = len(lats), len(lons), len(cells)
clat = cells.get_level_values("lat").to_numpy()
clon = cells.get_level_values("lon").to_numpy()
to_map = lambda v: np.asarray(v).reshape(nlat, nlon)          # flux vector -> 2D grid

blob = lambda la, lo, amp: amp * np.exp(-0.5 * (((clat - la) / 0.12) ** 2
                                                + ((clon - lo) / 0.12) ** 2))
x_true = 2.0 + blob(40.6, -111.7, 4.0) + blob(41.0, -112.1, 3.0)  # an SE source and an NW-corner source

# Observations: 35 footprints (rows of H), concentrated over the south and east,
# so the NW corner is barely seen.
rng = np.random.default_rng(1)
n_obs = 35
oa = rng.uniform(40.50, 40.85, n_obs)            # footprint centers (lat)
oo = rng.uniform(-111.95, -111.60, n_obs)        # footprint centers (lon)
H = np.array([np.exp(-0.5 * (((clat - a) / 0.13) ** 2 + ((clon - b) / 0.13) ** 2))
              for a, b in zip(oa, oo)])

obsid = pd.Index([f"obs_{i}" for i in range(n_obs)], name="obs")
H  = pd.DataFrame(H, index=obsid, columns=cells)
x0 = pd.Series(2.0, index=cells, name="flux")                      # flat prior
z  = pd.Series(H.values @ x_true + rng.normal(0, 0.1, n_obs), index=obsid, name="conc")
S0 = pd.DataFrame(np.eye(n) * 4.0, index=cells, columns=cells)     # prior error
Sz = pd.DataFrame(np.diag(np.full(n_obs, 0.04)), index=obsid, columns=obsid)  # obs error

prob = InverseProblem(obs=z, prior=x0, forward_operator=H,
                      prior_error=S0, modeldata_mismatch=Sz).solve("bayesian")

e = prob.estimator
print(f"DOFS = {e.DOFS:.1f} / {n},  R2 = {e.R2:.3f},  "
      f"uncertainty reduction = {e.uncertainty_reduction:.0%}")

In [ ]:
post = prob.posterior.to_series().values
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
for ax, (name, v) in zip(axes, [("truth", x_true), ("prior", x0.values), ("posterior", post)]):
    m = ax.pcolormesh(lons, lats, to_map(v), cmap="viridis", vmin=2, vmax=6, shading="nearest")
    ax.plot(oo, oa, "w.", ms=3, alpha=0.5)         # where the obs footprints are centered
    ax.set(title=name, xlabel="lon")
axes[0].set_ylabel("lat")
fig.colorbar(m, ax=axes, label="flux", fraction=0.025)
fig.suptitle("the inversion recovers the SE source the obs see; the NW corner stays near the prior")
plt.show()

The posterior is a distribution, not just a point estimate. Its covariance is
$\hat{S}$, so every cell has an uncertainty, and it shrinks from the prior exactly
where the observations constrain it (here, the south and east).

In [ ]:
post_sd = np.sqrt(np.diag(prob.posterior_error.values))    # posterior 1-sigma per cell

fig, ax = plt.subplots(figsize=(5.8, 4.2))
m = ax.pcolormesh(lons, lats, to_map(post_sd), cmap="cividis", shading="nearest")
ax.plot(oo, oa, "w.", ms=4, alpha=0.6, label="obs centers")
ax.set(title="posterior 1-sigma (the prior was 2.0 everywhere)", xlabel="lon", ylabel="lat")
ax.legend(loc="upper left"); fig.colorbar(m, ax=ax, label="flux 1-sigma")
plt.show()

That was the state-space view, flux per cell. The other half of any inversion is
the obs-space view: does the posterior actually reproduce the measurements? The
prior misses them; the posterior, by construction, fits.

In [ ]:
o   = prob.obs.to_series()
pri = prob.prior_obs.to_series()
pos = prob.posterior_obs.to_series()
r2_prior = np.corrcoef(o, pri)[0, 1] ** 2

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(o.values,   "ko", ms=9, label="observed z")
ax.plot(pri.values, "s", color="steelblue", label=f"prior  (R2 = {r2_prior:.2f})")
ax.plot(pos.values, "r^", label=f"posterior (R2 = {prob.estimator.R2:.2f})")
ax.set(xlabel="observation", ylabel="concentration",
       title="obs-space fit: does the posterior match the data?")
ax.legend(); plt.show()

One piece of the equation we have been setting aside: the background $c$. A real
tower measures enhancement on top of a large CH4 background (about 1900 ppb), so
the data are $z = Hx + c$. Hand $c$ to `fips` as `constant` and it inverts for the
fluxes; forget it and the background swamps the solution.

In [ ]:
# Background lives in observation (concentration) space, same block as the obs.
c = pd.Series(1900.0, index=obsid, name="conc")
z_bg = pd.Series(H.values @ x_true + c.values + rng.normal(0, 0.1, n_obs),
                 index=obsid, name="conc")          # what the tower actually measures

with_c = InverseProblem(obs=z_bg, prior=x0, forward_operator=H, prior_error=S0,
                        modeldata_mismatch=Sz, constant=c).solve("bayesian")
no_c   = InverseProblem(obs=z_bg, prior=x0, forward_operator=H, prior_error=S0,
                        modeldata_mismatch=Sz).solve("bayesian")   # background ignored

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
m1 = a1.pcolormesh(lons, lats, to_map(with_c.posterior.to_series().values),
                   cmap="viridis", vmin=2, vmax=6, shading="nearest")
a1.set(title="background passed as c (recovers the source)", xlabel="lon", ylabel="lat")
fig.colorbar(m1, ax=a1, fraction=0.046)
m2 = a2.pcolormesh(lons, lats, to_map(no_c.posterior.to_series().values),
                   cmap="viridis", shading="nearest")   # its own scale: the fluxes blow up
a2.set(title="background ignored (solution corrupted)", xlabel="lon")
fig.colorbar(m2, ax=a2, fraction=0.046)
plt.tight_layout(); plt.show()

Why did it recover the SE source but barely touch the NW corner? The averaging
kernel $A = KH$ answers it. Its diagonal is how much each cell's posterior comes
from the data versus the prior (1 fully observed, 0 unconstrained), and its trace
is the DOFS. Mapped back onto the grid, the uncertainty reduction is high right
where the observations are.

In [ ]:
A = prob.averaging_kernel.values
u_red = prob.estimator.U_red          # percent uncertainty reduction per cell

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
im = a1.imshow(A, cmap="RdBu_r", vmin=-1, vmax=1)
a1.set(title=f"averaging kernel A = KH  (trace = DOFS = {prob.estimator.DOFS:.1f})",
       xlabel="cell", ylabel="cell")
fig.colorbar(im, ax=a1, fraction=0.046)
m = a2.pcolormesh(lons, lats, to_map(u_red), cmap="Greens", shading="nearest")
a2.plot(oo, oa, "k.", ms=3, alpha=0.4)
a2.set(title="uncertainty reduction by cell (%)", xlabel="lon", ylabel="lat")
fig.colorbar(m, ax=a2, fraction=0.046)
plt.tight_layout(); plt.show()

One knob worth knowing: $\gamma$ scales how hard the solver trusts the data
relative to the prior (it divides $S_z$). Sweeping it traces the classic
trade-off, more data weight buys more degrees of freedom, until it overfits.

In [ ]:
gammas = [0.1, 0.3, 1.0, 3.0, 10.0]
swept = [InverseProblem(obs=z, prior=x0, forward_operator=H, prior_error=S0,
                        modeldata_mismatch=Sz).solve("bayesian", gamma=g).estimator
         for g in gammas]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gammas, [e.DOFS for e in swept], "o-", label="DOFS")
ax.plot(gammas, [e.uncertainty_reduction * 10 for e in swept], "s--",
        label="uncertainty reduction (x10)")
ax.set(xscale="log", xlabel=r"data weight $\gamma$", ylabel="diagnostic",
       title="more data weight, more signal (until it overfits)")
ax.legend(); plt.show()

The win for STILT work is that `fips` vectors and matrices are labeled and
blocked. Obs carry `(site, time)`, fluxes carry `(time, lat, lon)`, and the
covariances align by those labels automatically, so you reason in physical
coordinates instead of flat arrays.

That labeling is what makes physically-structured error covariances easy. Below,
a prior error covariance is built straight from the flux grid's coordinates, with
a correlation that decays with real (Haversine) distance, so nearby cells are
correlated and far ones are not. It drops straight into the problem as $S_0$, and
the same API scales from this toy to thousands of cells and observations.

In [ ]:
from fips.covariance import KroneckerError
from fips.kernels import GridSpatialDecay

# Same flux grid as the toy, but now the prior error is spatially correlated.
S_corr = KroneckerError(
    "prior_error", variances=4.0,
    marginal_kernels=[(["lat", "lon"], GridSpatialDecay("lat", "lon", scale=15.0))],
).build(cells)        # ~15 km correlation length

center = n // 2
corr_map = S_corr.iloc[center].values.reshape(nlat, nlon)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
im1 = a1.imshow(S_corr.values, cmap="viridis")
a1.set(title=f"prior covariance, {n} (lat,lon) cells")
fig.colorbar(im1, ax=a1, fraction=0.046)
m2 = a2.pcolormesh(lons, lats, corr_map, cmap="magma", shading="nearest")
a2.plot(lons[center % nlon], lats[center // nlon], "c*", ms=18)
a2.set(title="one cell's correlation, mapped", xlabel="lon", ylabel="lat")
fig.colorbar(m2, ax=a2, fraction=0.046)
plt.tight_layout(); plt.show()

And the blocks compose. Real inversions fuse observation networks with very
different error characteristics, say a continuous tower and a sparse satellite
column product. In `fips` each is a labeled block with its own forward operator and
error covariance; you hand both to one problem and the posterior is constrained by
both at once.

In [ ]:
from fips import Block, Vector, MatrixBlock

# Two observation networks over the same 6x6 flux grid.
tower = pd.MultiIndex.from_product(
    [pd.date_range("2023-01-01", periods=20, freq="D"), ["UOU"]], names=["time", "site"])
sat = pd.MultiIndex.from_product(
    [pd.date_range("2023-01-01", periods=4, freq="5D"), lats[::2], lons[::2]],
    names=["time", "lat", "lon"])

Ht = pd.DataFrame(rng.random((20, n)),        index=tower, columns=cells)
Hs = pd.DataFrame(rng.random((len(sat), n)),  index=sat,   columns=cells)
zt = pd.Series(Ht.values @ x_true + rng.normal(0, 0.2, 20),        index=tower)   # precise tower
zs = pd.Series(Hs.values @ x_true + rng.normal(0, 0.5, len(sat)),  index=sat)     # noisier satellite

prob_mb = InverseProblem(
    obs=Vector([Block(zt, name="tower"), Block(zs, name="satellite")]),
    prior=Vector(Block(pd.Series(2.0, index=cells), name="flux")),
    forward_operator=[MatrixBlock(Ht, row_block="tower", col_block="flux"),
                      MatrixBlock(Hs, row_block="satellite", col_block="flux")],
    prior_error=pd.DataFrame(np.eye(n) * 4.0, index=cells, columns=cells),
    modeldata_mismatch=[
        MatrixBlock(pd.DataFrame(np.diag([0.04] * 20), index=tower, columns=tower),
                    row_block="tower", col_block="tower"),
        MatrixBlock(pd.DataFrame(np.diag([0.25] * len(sat)), index=sat, columns=sat),
                    row_block="satellite", col_block="satellite")],
).solve("bayesian")

print(f"fused {prob_mb.n_obs} obs (20 tower + {len(sat)} satellite) -> DOFS {prob_mb.estimator.DOFS:.1f}")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
m1 = a1.pcolormesh(lons, lats, to_map(x_true), cmap="viridis", vmin=2, vmax=6, shading="nearest")
a1.set(title="truth", xlabel="lon", ylabel="lat"); fig.colorbar(m1, ax=a1, fraction=0.046)
m2 = a2.pcolormesh(lons, lats, to_map(prob_mb.posterior.to_series().values),
                   cmap="viridis", vmin=2, vmax=6, shading="nearest")
a2.set(title="posterior, fused from both networks", xlabel="lon")
fig.colorbar(m2, ax=a2, fraction=0.046)
plt.tight_layout(); plt.show()

---
# 4. `slv`: the pattern, wired into a real inversion

`slv` is the application all of this was built for. It is not on PyPI (it is the
science code for one valley, not a general tool), so we do not install it here.
But it is worth seeing, because it is short. `slv` subclasses a `fips` pipeline
and just says where each SLV-specific piece comes from. This is the actual code,
lightly trimmed:

```python
# slv/inversion/pipelines.py
from fips.problems.flux import FluxInversionPipeline, JacobianBuilder
from stilt import Model

class SLVMethaneInversion(FluxInversionPipeline):

    def get_obs(self):                           # tower CH4 enhancements   -> z
        return get_slv_observations(self.config.sites, self.config.time_range, ...)

    def get_prior(self):                         # an EPA inventory         -> x0
        return get_slv_prior(self.config.prior, self.config.grid, ...)

    def get_forward_operator(self, obs, prior):  # pystilt footprints       -> H
        model = Model(self.config.stilt_project)
        return JacobianBuilder(model).build_from_grid(
            self.config.grid, flux_times=self.config.flux_time_bins,
            footprint=self.config.footprint, ...)

    def get_constant(self, obs):                 # CarbonTracker at endpoints -> c
        return get_slv_background(self.config.background, obs, ...)

    def get_prior_error(self, prior):            # spatial/temporal corr.   -> S_0
        return build_prior_error(prior, time_scale=..., spatial_scale=...)

    def get_modeldata_mismatch(self, obs):       # error budget             -> S_z
        return build_mdm_error(obs, self.config.mdm_components, ...)
```

You wire it together with a config. A subset of the real `InversionConfig`:

```python
from slv.inversion import InversionConfig, SLVMethaneInversion

config = InversionConfig(
    tstart="2016-01-01", tend="2023-01-01",   # study period
    sites=["uou"],                            # which tower(s) supply the obs
    subset_hours=[12, 13, 14, 15, 16],        # well-mixed local afternoon only
    flux_freq="MS", dx=0.05, dy=0.05,         # monthly fluxes on a 0.05 degree grid
    prior="epa",                              # EPA inventory as the prior x0
    background="ct_stilt",                    # CarbonTracker-STILT background c
    stilt_project="/path/to/stilt",           # where the pystilt footprints live
    prior_time_scale="32d", prior_spatial_scale=5.0,   # prior-error correlation
    gamma=1.0,                                # data-vs-prior weight
)
```

Each `get_*` is one of the packages from this tour: `get_obs` is the measurements,
`get_forward_operator` is pystilt footprints turned into the Jacobian $H$,
`get_constant` is a CarbonTracker background sampled at the trajectory endpoints,
and the `fips` base class collects the six pieces, builds the `FluxProblem`, and
solves it. So the whole SLV inversion is one line:

```python
problem = SLVMethaneInversion(config).run()   # a solved fips FluxProblem
fluxes  = problem.posterior_fluxes            # CH4 flux per grid cell and month
```

The one piece worth running live is the heart of `get_forward_operator`: turning a
footprint into a row of $H$. `JacobianBuilder` does exactly this for every
observation, and on our one footprint it is the `aggregate` from earlier:

In [ ]:
# Regrid the footprint onto the inversion's (coarser) flux grid and integrate over
# its time window. aggregate() does an area-conserving regrid, so the result is
# exactly one row of the Jacobian H: this observation's sensitivity to each flux cell.
flux_cells = [(round(x, 2), round(y, 2)) for x in np.arange(-112.4, -111.45, 0.1)
                                          for y in np.arange(40.45, 41.05, 0.1)]
t0, t1 = foot.time_range
window = pd.IntervalIndex.from_breaks(pd.to_datetime(
    [pd.Timestamp(t0).tz_localize(None) - pd.Timedelta("1h"),
     pd.Timestamp(t1).tz_localize(None) + pd.Timedelta("1h")]))
H_row = foot.aggregate(flux_cells, window, resolution=0.1).iloc[:, 0]

modeled_ppm = float((H_row * 1.0).sum())   # times a uniform 1 umol m-2 s-1 flux
print(f"one footprint -> one row of H over {H_row.size} flux cells "
      f"({int((H_row > 0).sum())} of them nonzero)")
print(f"that row applied to a uniform 1 umol m-2 s-1 flux models a "
      f"{modeled_ppm:.3f} ppm enhancement")
print("stack one row per observation, add a prior, a background, and error budgets")

---
# Takeaways

- One ecosystem from met to flux. `arlmet`, `pystilt`, and `fips` are small,
  composable, and `xarray` and `pandas` native, so there is no format-shuffling
  between stages.
- The footprint is the bridge between transport and inversion. In Python the
  Jacobian is one `xarray` operation away.
- Labeled, blocked vectors keep spatiotemporal metadata aligned, so the
  inverse-problem code reads like the physics.
- The same APIs scale from a 10-cell teaching toy to the full SLV CH4 inversion.

Source on GitHub at `jmineau`: `arl-met`, `PYSTILT`, `fips`, `slv`. Demo met from
`uataq/stilt-tutorials`.

Thanks. james.mineau@utah.edu